In [1]:
import pandas as pd
import logging
from contextlib import redirect_stdout
import io
import sys
import os
os.getcwd()

'c:\\Users\\user\\OneDrive\\Desktop\\데이터마케팅\\효과예측 (외부)\\thedap_api\\src'

In [2]:
# pip install flask
import warnings
import json
from urllib.parse import quote
from datetime import datetime, date, timedelta
from io import BytesIO
warnings.filterwarnings(action='ignore')

from CONFIG.DapData import DapData
from THEDAP_UTILS.DapUtils_v5 import DapUtils_v5

from THEDAP_SIMULATION.DapOutput_v4 import DapOutput_v4
from THEDAP_REACHCURVE.DapCurve_v4 import DapCurve_v4

from THEDAP_SIMULATION.DapOutput_v5 import DapOutput_v5
from THEDAP_REACHCURVE.DapCurve_v5 import DapCurve_v5
from THEDAP_COPULA.DapCopula import DapCopula
from THEDAP_MIXOPTIM.DapMixOptimizer import DapMixOptimizer
from THEDAP_UTILS.DapCustomModel import DapCustomModel
from THEDAP_REPORT import *

In [3]:
#### target_info

In [4]:
with open("DATA_SAMPLES/0101_target_info.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [6]:
gender = data["input_gender"]
age_min = data["input_age_min"]
age_max = data["input_age_max"]
modelDate = data.get("inputModelDate", datetime.strftime(date.today(), "%Y-%m-%d"))

obj_ = DapUtils_v5(inputModelDate=modelDate, pop_only=True)
input_gender = json.dumps([{"input_gender": gender}])
input_age = json.dumps([{"input_age_min": age_min, "input_age_max":age_max}])

result = obj_.get_target_info(input_gender, input_age)

In [7]:
#### reach_result

In [8]:
with open("DATA_SAMPLES/0401_reach_result.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

with open("DATA_SAMPLES/0402_reach_result.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [20]:
data = {
    "input_mix": [
        {
            "campaign": "1안",
            "platform": "Youtube",
            "product": "건너뛸수없는광고",
            "date_start": "",
            "date_end": "",
            "gender": "P",
            "min": 18,
            "max": 54,
            "retargeting": "",
            "impact": 1,
            "budget": 400000000,
            "bid_type": "E.IMP 직접입력",
            "bid_cost": "",
            "bid_rate": 0,
            "imp": 33333333,
            "reach": ""
        },
        {
            "campaign": "1안",
            "platform": "Instagram",
            "product": "AUCTION_VIDEO_VIEWS",
            "date_start": "",
            "date_end": "",
            "gender": "P",
            "min": 18,
            "max": 54,
            "retargeting": "",
            "impact": 1,
            "budget": 238000000,
            "bid_type": "E.IMP 직접입력",
            "bid_cost": "",
            "bid_rate": 0,
            "imp": 13222222,
            "reach": ""
        },
        {
            "campaign": "1안",
            "platform": "Instagram",
            "product": "AUCTION_REACH",
            "date_start": "",
            "date_end": "",
            "gender": "P",
            "min": 18,
            "max": 54,
            "retargeting": "",
            "impact": 1,
            "budget": 102000000,
            "bid_type": "E.IMP 직접입력",
            "bid_cost": "",
            "bid_rate": 0,
            "imp": 20400000,
            "reach": ""
        },
        {
            "campaign": "1안",
            "platform": "GFA",
            "product": "브랜드 메인/서브",
            "date_start": "",
            "date_end": "",
            "gender": "P",
            "min": 18,
            "max": 54,
            "retargeting": "",
            "impact": 1,
            "budget": 100000000,
            "bid_type": "E.IMP 직접입력",
            "bid_cost": "",
            "bid_rate": 0,
            "imp": 33333333,
            "reach": ""
        },
        {
            "campaign": "1안",
            "platform": "GFA",
            "product": "브랜드 메인/서브",
            "date_start": "",
            "date_end": "",
            "gender": "P",
            "min": 18,
            "max": 54,
            "retargeting": "",
            "impact": 1,
            "budget": 100000000,
            "bid_type": "E.IMP 직접입력",
            "bid_cost": "",
            "bid_rate": 0,
            "imp": 8333333,
            "reach": ""
        }
    ],
    "input_weight": "basic",
    "input_gender": "P",
    "input_age_min": 18,
    "input_age_max": 54,
    "userGrade": "P",
    "userName": "admin@sbs.co.kr",
    "inputModelDate": "2026-01-02"
}

In [21]:
mix = data["input_mix"]
gender = data["input_gender"]
age_min = data["input_age_min"]
age_max = data["input_age_max"]
weight = data["input_weight"]

# 사용자 정보 & 모델버전
userGrade = data["userGrade"]
userName = data.get("userName", "")
modelDate = data.get("inputModelDate", datetime.strftime(date.today(), "%Y-%m-%d"))

input_mix = json.dumps(mix)
input_gender = json.dumps([{"input_gender": gender}])
input_age = json.dumps([{"input_age_min": age_min, "input_age_max":age_max}])
input_weight = json.dumps([{"input_weight": weight}])
platform_list=list(set(pd.read_json(input_mix)['platform']))

In [22]:
age_min, age_max, gender, weight

(18, 54, 'P', 'basic')

In [23]:
# 로깅
buf = io.StringIO()
with redirect_stdout(buf):
    if userGrade == 'B':
        thedap_output = DapOutput_v4(
            input_mix, input_age, input_gender, input_weight
        )
        
        summary_ = thedap_output.result_summary()
        result = {
            "result_overall": thedap_output.result_overall(),
            "result_summary": summary_,
            "heatmap": thedap_output.heatmap(),
            "reach_freq": thedap_output.reach_freq()
        }
        
    else:            
        thedap_output = DapOutput_v5(
            input_mix, input_age, input_gender, input_weight, 
            userName=userName, inputModelDate=modelDate,
            platform_list=platform_list
        )
        
        summary_ = thedap_output.result_summary()
        result = {
            "result_overall": thedap_output.result_overall(),
            "result_summary": summary_,
            "heatmap": thedap_output.heatmap(),
            "reach_freq": thedap_output.reach_freq(),
            "reach_marginal": {line['platform']:line['target_reach_p'] for line in summary_ if line['line'] == "Platform Total"},
            "reach_union": [line['target_reach_p'] for line in summary_ if line['line'] == "Total"][0]
        }

In [24]:
result['result_overall']

[{'R1_p': 0.734901,
  'R3_p': 0.509956,
  'GRPs': 631.503466,
  'AF': 8.593046,
  'R1_n': 12640713.0,
  'R3_n': 8771536.0}]

In [14]:
pd.DataFrame(result['result_summary'])

,line,campaign,platform,product,gender,age_min,age_max,budget,date_start,date_end,...,target_reach7_p,target_reach8_p,target_reach9_p,target_reach10_p,target_af,target_af_weighted,target_af_day,target_af_week,target_af_30,imp_interval
0,01,2안,Youtube,건너뛸수없는광고,P,19.0,54.0,4.000000e+08,2026-07-06,2026-08-04,...,0.048375,0.037584,0.029682,0.023524,3.348245,3.348245,0.111608,0.781257,3.348245,8.96일당 1회 노출
1,Sub Total,2안,Youtube,None,None,NaN,NaN,4.000000e+08,2026-07-06,2026-08-04,...,0.048375,0.037584,0.029682,0.023524,3.348245,3.348245,0.111608,0.781257,3.348245,8.96일당 1회 노출
2,02,2안,Instagram,AUCTION_VIDEO_VIEWS,P,19.0,54.0,2.380000e+08,2026-07-06,2026-08-04,...,0.016565,0.012687,0.009875,0.007801,3.276180,3.276180,0.109206,0.764442,3.276180,9.16일당 1회 노출
3,03,2안,Instagram,AUCTION_REACH,P,19.0,54.0,1.020000e+08,2026-07-06,2026-08-04,...,0.022993,0.016856,0.012631,0.009521,2.589541,2.589541,0.086318,0.604226,2.589541,11.59일당 1회 노출
4,Sub Total,2안,Instagram,None,None,NaN,NaN,3.400000e+08,2026-07-06,2026-08-04,...,0.051857,0.041101,0.033045,0.026690,3.732277,3.732277,0.124409,0.870865,3.732277,8.04일당 1회 노출
5,04,2안,GFA,브랜드 메인/서브,P,19.0,54.0,1.000000e+08,2026-07-06,2026-08-04,...,0.051418,0.040771,0.032782,0.026489,3.772563,3.772563,0.125752,0.880265,3.772563,7.95일당 1회 노출
6,05,2안,GFA,브랜드 메인/서브,P,19.0,54.0,1.000000e+08,2026-07-06,2026-08-04,...,0.006807,0.004726,0.003362,0.002432,2.237122,2.237122,0.074571,0.521995,2.237122,13.41일당 1회 노출
7,Sub Total,2안,GFA,None,None,NaN,NaN,2.000000e+08,2026-07-06,2026-08-04,...,0.067966,0.054626,0.044477,0.036337,4.021769,4.021769,0.134059,0.938413,4.021769,7.46일당 1회 노출
8,06,2안,Tving,overall,P,7.0,79.0,1.680000e+08,2026-07-06,2026-08-04,...,0.006970,0.005538,0.004447,0.003666,4.212079,4.212079,0.140403,0.982818,4.212079,7.12일당 1회 노출
9,07,2안,Tving,overall,P,7.0,79.0,7.200000e+07,2026-07-06,2026-08-04,...,0.000796,0.000564,0.000407,0.000309,2.603378,2.603378,0.086779,0.607455,2.603378,11.52일당 1회 노출


In [16]:
pd.DataFrame(result['result_summary']).to_excel('result_summary_2.xlsx', index=False)

In [32]:
#### mix_sample

In [21]:
with open("DATA_SAMPLES/0301_mix_sample.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [ ]:
userGrade = data.get("userGrade") # "B"
userName = data.get("userName", "")
channelVehicleList = data.get("list")
channelVehicleDf = pd.DataFrame(channelVehicleList).groupby('platform').\
    agg(vehicle = ('product', lambda x: list(x))).reset_index()
channelVehicleMap = {row['platform']:row['vehicle'] for _, row in channelVehicleDf.iterrows()}

# 사용자 등급에 따른 미디어믹스 양식 파일 생성
mix_wb = DapMixSample(channelVehicleMap, userGrade=userGrade)

In [21]:
result

{'result_overall': [{'R1_p': 0.670435,
   'R3_p': 0.554919,
   'GRPs': 1142.023776,
   'AF': 17.034077,
   'R1_n': 6376760.0,
   'R3_n': 5278045.0}],
 'result_summary': [{'line': '01',
   'campaign': '1안',
   'platform': 'Youtube',
   'product': '건너뛸수없는광고',
   'gender': 'P',
   'age_min': 19,
   'age_max': 54,
   'budget': 400000000.0,
   'date_start': '2026-07-03',
   'date_end': '2026-08-01',
   'target_impression': 33333333.0,
   'target_impression_weighted': 33333333.0,
   'target_grps': 350.457378,
   'target_grps_weighted': 350.457378,
   'target_reach_n': 5663144.0,
   'target_reach2_n': 4214618.0,
   'target_reach3_n': 3327992.0,
   'target_reach4_n': 2710829.0,
   'target_reach5_n': 2252108.0,
   'target_reach6_n': 1895747.0,
   'target_reach7_n': 1611608.0,
   'target_reach8_n': 1378576.0,
   'target_reach9_n': 1188719.0,
   'target_reach10_n': 1026263.0,
   'target_reach_p': 0.595407,
   'target_reach2_p': 0.443113,
   'target_reach3_p': 0.349896,
   'target_reach4_p': 0.285

In [26]:
mix_wb.save("미디어믹스_샘플_(260205).xlsx")

In [27]:
#### report_reach

In [19]:
# with open("DATA_SAMPLES/0501_report_analysis.json", 'r', encoding='utf-8') as js:
#     data = json.load(js)

with open("DATA_SAMPLES/0502_report_analysis.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [22]:
result

{'result_overall': [{'R1_p': 0.670435,
   'R3_p': 0.554919,
   'GRPs': 1142.023776,
   'AF': 17.034077,
   'R1_n': 6376760.0,
   'R3_n': 5278045.0}],
 'result_summary': [{'line': '01',
   'campaign': '1안',
   'platform': 'Youtube',
   'product': '건너뛸수없는광고',
   'gender': 'P',
   'age_min': 19,
   'age_max': 54,
   'budget': 400000000.0,
   'date_start': '2026-07-03',
   'date_end': '2026-08-01',
   'target_impression': 33333333.0,
   'target_impression_weighted': 33333333.0,
   'target_grps': 350.457378,
   'target_grps_weighted': 350.457378,
   'target_reach_n': 5663144.0,
   'target_reach2_n': 4214618.0,
   'target_reach3_n': 3327992.0,
   'target_reach4_n': 2710829.0,
   'target_reach5_n': 2252108.0,
   'target_reach6_n': 1895747.0,
   'target_reach7_n': 1611608.0,
   'target_reach8_n': 1378576.0,
   'target_reach9_n': 1188719.0,
   'target_reach10_n': 1026263.0,
   'target_reach_p': 0.595407,
   'target_reach2_p': 0.443113,
   'target_reach3_p': 0.349896,
   'target_reach4_p': 0.285

In [20]:
userGrade = data.get("userGrade")
reportOption = data.get("reportOption")

if not reportOption.get("inputModelDate"):
    reportOption['inputModelDate'] = datetime.strftime(date.today(), "%Y-%m-%d")

reportResult = data.get("reportResult")
if reportResult.get("heatmap"):
    heatmap_re = {}
    for h in reportResult['heatmap']:
        heatmap_re[h['name']] = [
            {
                'e_reach_p':h['P']
            },
            {
                'e_reach_n':h['N']
            },
            {
                "e_grps":h['GRP']
            }
        ]
    reportResult["heatmap"] = [heatmap_re]
    
target_pop = data.get("target_pop")

report_wb = DapReportReachAnalysis(reportOption, reportResult, target_pop, userGrade)


In [38]:
report_wb.save("통합 Reach 분석결과_(260206).xlsx")

In [39]:
#### reach_copula

In [40]:
with open("DATA_SAMPLES/0601_reach_copula.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [41]:
marginal_probs = data.get("reach_marginal")
union_obs = data.get("reach_union")

DC = DapCopula(marginal_probs, union_obs)
u, i = DC.getCopulaProbs(marginal_probs, union_obs)

result = {
    'copula_union': u,
    'copula_inter': i
}

In [43]:
#### report_copula

In [44]:
with open("DATA_SAMPLES/0701_report_copula.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [45]:
reportOption = data.get("reportOption")
reportCopula = data.get("reportCopula")
target_pop = data.get("target_pop")

report_wb = DapReportCopula(reportOption, reportCopula, target_pop)

In [47]:
report_wb.save("매체간 중복&통합 분석결과_(260206).xlsx")

In [48]:
#### reach_curve

In [50]:
with open("DATA_SAMPLES/0801_reach_curve.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [79]:
data = {"input_mix":[{"campaign":"","platform":"Instagram","product":"AUCTION_VIDEO_VIEWS","date_start":"2026-05-11","date_end":"2026-05-29","gender":"P","min":20,"max":49,"retargeting":"","impact":"","budget":5030301,"bid_type":"E.IMP 직접입력","bid_cost":"","bid_rate":"","imp":437417,"reach":""}],"input_weight":"high","input_gender":"P","input_age_min":20,"input_age_max":49,"input_maxbudget":100,"input_seq":10,"userGrade":"P","userName":"ma5@dmcmedia.co.kr","inputModelDate":"2026-01-02"}


In [80]:
mix = data["input_mix"]
gender = data["input_gender"]
age_min = data["input_age_min"]
age_max = data["input_age_max"]
weight = data["input_weight"]
maxbudget = data["input_maxbudget"]
seq = data["input_seq"]

# 사용자 정보 & 모델버전
userGrade = data["userGrade"]
userName = data.get("userName", "")
modelDate = data.get("inputModelDate", datetime.strftime(date.today(), "%Y-%m-%d"))

input_mix = json.dumps(mix)
input_gender = json.dumps([{"input_gender": gender}])
input_age = json.dumps([{"input_age_min": age_min, "input_age_max":age_max}])
input_weight = json.dumps([{"input_weight": weight}])
input_maxbudget = json.dumps([{"input_maxbudget": maxbudget}])
input_seq = json.dumps([{"input_seq": seq}])
platform_list=list(set(pd.read_json(input_mix)['platform']))

if userGrade == 'B':
    rc = DapCurve_v4()
else:
    rc = DapCurve_v5(
        userName=userName, inputModelDate=modelDate, 
        platform_list=platform_list
    )

result = rc.reach_curve(input_mix, input_age, input_gender, input_weight, input_seq, input_maxbudget)

In [81]:
pd.DataFrame(result)

,idx,budget,target_grps,target_grps_weighted,target_reach_p,target_reach2_p,target_reach3_p,target_reach4_p,target_reach5_p,target_reach6_p,target_reach7_p,target_reach8_p,target_reach9_p,target_reach10_p,target_af
0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1.0,10000000.0,4.116499,4.116499,0.023817,0.008565,0.003770,0.001945,0.001074,0.000641,0.000392,0.000246,0.000158,0.000108,1.728396
2,2.0,20000000.0,8.232998,8.232998,0.042189,0.017024,0.008266,0.004587,0.002709,0.001708,0.001100,0.000724,0.000489,0.000344,1.951446
3,3.0,30000000.0,12.349498,12.349498,0.058418,0.025249,0.012984,0.007525,0.004622,0.003009,0.001999,0.001354,0.000940,0.000676,2.114003
4,4.0,40000000.0,16.465997,16.465997,0.073178,0.033245,0.017811,0.010652,0.006730,0.004484,0.003047,0.002107,0.001492,0.001088,2.250132
5,5.0,50000000.0,20.582496,20.582496,0.086804,0.041021,0.022693,0.013912,0.008987,0.006099,0.004218,0.002966,0.002133,0.001575,2.371150
6,6.0,60000000.0,24.698995,24.698995,0.099501,0.048588,0.027600,0.017271,0.011364,0.007830,0.005496,0.003918,0.002855,0.002129,2.482294
7,7.0,70000000.0,28.815494,28.815494,0.111410,0.055956,0.032510,0.020705,0.013838,0.009662,0.006867,0.004955,0.003650,0.002747,2.586441
8,8.0,80000000.0,32.931994,32.931994,0.122636,0.063134,0.037410,0.024196,0.016395,0.011580,0.008322,0.006068,0.004514,0.003424,2.685351
9,9.0,90000000.0,37.048493,37.048493,0.133259,0.070130,0.042288,0.027731,0.019021,0.013574,0.009852,0.007251,0.005441,0.004157,2.780183


In [55]:
#### report_curve

In [56]:
with open("DATA_SAMPLES/0901_report_curve.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [57]:
reportOption = data.get("reportOption")
reportCurve = data.get("reportCurve")
target_pop = data.get("target_pop")

report_wb = DapReportReachCurve(reportOption, reportCurve, target_pop)
report_wb.save('리치커브 분석결과_(260206).xlsx')

<br>

#### 최적화

In [79]:
# with open("DATA_SAMPLES/1001_reach_optimize.json", 'r', encoding='utf-8') as js:
#     data = json.load(js)
    
# with open("DATA_SAMPLES/1002_reach_optimize.json", 'r', encoding='utf-8') as js:
#     data = json.load(js)
    
with open("DATA_SAMPLES/1003_reach_optimize.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [80]:
type = data.get("opt_type")
# 사용자 정보 & 모델버전
userName = ""#data.get("userName", "")
modelDate = data.get("inputModelDate", datetime.strftime(date.today(), "%Y-%m-%d"))

# REACH MAX
if type == "reach_max":
    mix = data["input_mix"]
    age_min = data["input_age_min"]
    age_max = data["input_age_max"]
    gender = data["input_gender"]
    weight = data["input_weight"]
    maxbudget = data["opt_maxbudget"]
    seq = data["opt_seq"]
                
    opt_type = json.dumps([{"opt_type": type}])
    opt_mix = json.dumps(mix)
    input_age = json.dumps([{"input_age_min": age_min, "input_age_max": age_max}])
    input_gender = json.dumps([{"input_gender": gender}])
    input_weight = json.dumps([{"input_weight": weight}])
    opt_maxbudget = json.dumps([{"opt_maxbudget": maxbudget}])
    opt_seq = json.dumps([{"opt_seq": seq}])
    platform_list = list(set(pd.read_json(opt_mix)['platform']))

    optimizer = DapMixOptimizer(
        opt_type=opt_type,
        opt_mix=opt_mix,
        input_age=input_age,
        input_gender=input_gender,
        input_weight=input_weight,
        opt_maxbudget=opt_maxbudget,
        opt_seq=opt_seq,
        userName=userName,
        inputModelDate=modelDate,
        platform_list=platform_list
    )

    result = optimizer.get_result()

# REACH TARGET
elif type == "reach_target":
    mix = data["input_mix"]
    age_min = data["input_age_min"]
    age_max = data["input_age_max"]
    gender = data["input_gender"]
    weight = data["input_weight"]
    target = data["opt_target"]
        
    opt_type = json.dumps([{"opt_type": type}])
    opt_mix = json.dumps(mix)
    input_age = json.dumps([{"input_age_min": age_min, "input_age_max": age_max}])
    input_gender = json.dumps([{"input_gender": gender}])
    input_weight = json.dumps([{"input_weight": weight}])
    opt_target = json.dumps([{"opt_target": target}])
    platform_list = list(set(pd.read_json(opt_mix)['platform']))
    
    checker = DapUtils_v5()
    chkFlag = checker.check_coverage(opt_mix, opt_target, input_age, input_gender)
    if chkFlag :
        optimizer = DapMixOptimizer(
            opt_type=opt_type,
            opt_mix=opt_mix,
            input_age=input_age,
            input_gender=input_gender,
            input_weight=input_weight,
            opt_target=opt_target,
            userName=userName,
            inputModelDate=modelDate,
            platform_list=platform_list
        )

        result = optimizer.get_result()
        result['isSuc'] = True
    else:
        result = {"isSuc":False}

elif type == "reach_spectrum":
    mixA = data["input_mixA"]
    mixB = data["input_mixB"]
    age_min = data["input_age_min"]
    age_max = data["input_age_max"]
    gender = data["input_gender"]
    weight = data["input_weight"]
    maxbudget = data["opt_maxbudget"]
    seq = data["opt_seq"]
    
    opt_type = json.dumps([{"opt_type": type}])
    opt_mix = [{"mix_a": mixA, "mix_b": mixB}]
    input_age = json.dumps([{"input_age_min": age_min, "input_age_max": age_max}])
    input_gender = json.dumps([{"input_gender": gender}])
    input_weight = json.dumps([{"input_weight": weight}])
    opt_maxbudget = json.dumps([{"opt_maxbudget": maxbudget}])
    opt_seq = json.dumps([{"opt_seq": seq}])
    platform_list = list(set([l['platform'] for l in mixA] + [l['platform'] for l in mixB]))

    optimizer = DapMixOptimizer(
        opt_type=opt_type,
        opt_mix=opt_mix,
        input_age=input_age,
        input_gender=input_gender,
        input_weight=input_weight,
        opt_maxbudget=opt_maxbudget,
        opt_seq=opt_seq,
        userName=userName,
        inputModelDate=modelDate,
        platform_list=platform_list
    )

    result = optimizer.get_result()

In [82]:
# print(json.dumps(result, indent=2, ensure_ascii=False))

In [83]:
#### report optimize

In [125]:
# with open("DATA_SAMPLES/1101_report_optimize.json", 'r', encoding='utf-8') as js:
#     data = json.load(js)
    
# with open("DATA_SAMPLES/1102_report_optimize.json", 'r', encoding='utf-8') as js:
#     data = json.load(js)
    
with open("DATA_SAMPLES/1103_report_optimize.json", 'r', encoding='utf-8') as js:
    data = json.load(js)

In [ ]:
reportOption = data.get("reportOption")
opt_type = reportOption.get('opt_type')
opt_type_kr = '도달 극대화' if opt_type == "reach_max" else ('도달률 달성' if opt_type == "reach_target" else "도달 스펙트럼")

reportOptimize = data.get("reportOptimize")
target_pop = data.get("target_pop")

if opt_type == ""

In [135]:
report_wb = DapReportReachSpectrum(reportOption, reportOptimize, target_pop)
report_wb.save('미디어믹스 최적화 분석결과 (도달 스펙트럼, 260206).xlsx')

In [6]:
#### 커스텀 모델 데이터양식

In [5]:
wb = DapCustomSample()
wb.save("커스텀 모델_데이터양식.xlsx")

In [1]:
import pandas as pd

In [17]:
param_xlsx = pd.read_excel('./DATA_BKUP/db_param_260317.xlsx')

In [18]:
param_xlsx.groupby(['PLATFORM', 'PRODUCT', 'GENDER', 'AGE_MIN', 'AGE_MAX','BASIS_DT']).\
    agg(row_cnt = ('BASIS_DT', 'count')).reset_index().query('row_cnt != 1')

,PLATFORM,PRODUCT,GENDER,AGE_MIN,AGE_MAX,BASIS_DT,row_cnt
17912,TV,지상파,F,50,54,20230713,2
17918,TV,지상파,F,55,59,20230713,2
17930,TV,지상파,F,65,69,20230713,2
17936,TV,지상파,F,70,74,20230713,2
17942,TV,지상파,F,75,79,20230713,2


In [19]:
param_xlsx.groupby(['BASIS_DT', 'PLATFORM', 'PRODUCT', 'GENDER', ]).\
    agg(row_cnt = ('BASIS_DT', 'count')).reset_index().query('row_cnt != 14')

,BASIS_DT,PLATFORM,PRODUCT,GENDER,row_cnt
78,20230713,TV,지상파,F,19
79,20230713,TV,지상파,M,9
